In [56]:
import pandas as pd
from pathlib import Path
import re


In [16]:
BASE_DIR = Path.cwd().parent
CHAMPIONS_LEAGUE_DATA_DIR = BASE_DIR / "data" / "raw" / "champions_league"
df = pd.read_csv(CHAMPIONS_LEAGUE_DATA_DIR / "2021_22_main_stages.csv")

In [17]:
df.head()

,Round,Wk,Day,Date,Time,Home,Score,Away,Attendance,Venue,Referee,Match Report,Notes
0,Group stage,1.0,Tue,2021-09-14,18:45,Sevilla es,1–1,at RB Salzburg,18373.0,Estadio Ramón Sánchez Pizjuán,Aliaksei Kulbakou,Match Report,NaN
1,Group stage,1.0,Tue,2021-09-14,18:45,Young Boys ch,2–1,eng Manchester Utd,31120.0,Wankdorfstadion,François Letexier,Match Report,NaN
2,Group stage,1.0,Tue,2021-09-14,20:00 (21:00),Chelsea eng,1–0,ru Zenit,39252.0,Stamford Bridge,Bartosz Frankowski,Match Report,NaN
3,Group stage,1.0,Tue,2021-09-14,21:00,Villarreal es,2–2,it Atalanta,12916.0,Estadio de la Cerámica,Clément Turpin,Match Report,NaN
4,Group stage,1.0,Tue,2021-09-14,21:00,Barcelona es,0–3,de Bayern Munich,39737.0,Camp Nou,Michael Oliver,Match Report,NaN


In [20]:
champions_league_file_list = list(CHAMPIONS_LEAGUE_DATA_DIR.iterdir())
print(f"Wszystkie pliki i foldery: {files}")

Wszystkie pliki i foldery: [WindowsPath('D:/Python Projects/UEFA_coefficient/data/raw/champions_league/2021_22_main_stages.csv'), WindowsPath('D:/Python Projects/UEFA_coefficient/data/raw/champions_league/2021_22_qualification.csv'), WindowsPath('D:/Python Projects/UEFA_coefficient/data/raw/champions_league/2022_23_main_stages.csv'), WindowsPath('D:/Python Projects/UEFA_coefficient/data/raw/champions_league/2022_23_qualification.csv'), WindowsPath('D:/Python Projects/UEFA_coefficient/data/raw/champions_league/2023_24_main_stages.csv'), WindowsPath('D:/Python Projects/UEFA_coefficient/data/raw/champions_league/2023_24_qualification.csv'), WindowsPath('D:/Python Projects/UEFA_coefficient/data/raw/champions_league/2024_25_main_stages.csv'), WindowsPath('D:/Python Projects/UEFA_coefficient/data/raw/champions_league/2024_25_qualification.csv'), WindowsPath('D:/Python Projects/UEFA_coefficient/data/raw/champions_league/2025_26_main_stages.csv'), WindowsPath('D:/Python Projects/UEFA_coefficie

In [63]:
#checking wheter structure of files are the same across years
df_champions_league =[]
for file in files:
    df_file = pd.read_csv(file)
    print(list(df_file.columns))
    print(f"{file} shape -{df_file.shape}")



['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\champions_league\2021_22_main_stages.csv shape -(131, 13)
['Round', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\champions_league\2021_22_qualification.csv shape -(98, 12)
['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\champions_league\2022_23_main_stages.csv shape -(131, 13)
['Round', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\champions_league\2022_23_qualification.csv shape -(94, 12)
['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match R

In [77]:
# main stages has more columns, the rest column have the same naming, so UNIONing is possible
# additional columns are needed for easier sorting / division of the data
dfs = []
for file in files:
    df_file = pd.read_csv(file)
    filename = str(file).split("\\")[-1]
    competition_phase = ''
    if re.search('main_stages',filename):
        competition_phase = 'main stage'
    elif re.search('qualification',filename):
        competition_phase = 'qualification'
    else:
        'N/A'
        # LOG ERROR HERE AND BREAK CODE !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        pass

    season = filename.replace("_","/")[:7]
    season_start = filename[:4]
    df_file['competition_phase'] = competition_phase
    df_file['season'] = season
    df_file['season_start'] = season_start
    dfs.append(df_file)

df_champions_league = pd.concat(dfs, ignore_index=True)

In [78]:
### further data cleaning
df_champions_league['Time'] = df_champions_league['Time'].apply(lambda x: str(x).split(' ')[0])

In [121]:
### data enhancment

# penalties addition
df_champions_league['extra time'] = df_champions_league['Notes'].str.contains('extra time', case=False, regex=False)
df_champions_league['penalty kicks'] = df_champions_league['Notes'].str.contains('penalty shoot-out', regex=False)
#re.search("penalty shoot-out",df_champions_league['
penalties = df_champions_league['Score'].str.findall(r'\((\d+)\)')

df_champions_league['Host penalties'] = penalties.apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else None)
df_champions_league['Guest penalties'] = penalties.apply(lambda x: x[-1] if isinstance(x, list) and len(x) >= 2 else None)

In [129]:
result_pattern = r'(?:\(\d+\)\s*)?(\d+)\s*[–-]\s*(\d+)(?:\s*\(\d+\))?'
result = df_champions_league['Score'].str.extract(pattern)
df_champions_league['Host goals'] = pd.to_numeric(result[0]).astype('Int64')
df_champions_league['Guest goals'] = pd.to_numeric(result[1]).astype('Int64')

In [130]:
df_champions_league

,Round,Wk,Day,Date,Time,Home,Score,Away,Attendance,Venue,...,season,season_start,Host,Host Country,Host penalties,Guest penalties,penalty kicks,extra time,Host goals,Guest goals
0,Group stage,1.0,Tue,2021-09-14,18:45,Sevilla es,1–1,at RB Salzburg,18373.0,Estadio Ramón Sánchez Pizjuán,...,2021/22,2021,Sevilla,es,NaN,NaN,False,False,1,1
1,Group stage,1.0,Tue,2021-09-14,18:45,Young Boys ch,2–1,eng Manchester Utd,31120.0,Wankdorfstadion,...,2021/22,2021,Young Boys,ch,NaN,NaN,False,False,2,1
2,Group stage,1.0,Tue,2021-09-14,20:00,Chelsea eng,1–0,ru Zenit,39252.0,Stamford Bridge,...,2021/22,2021,Chelsea,eng,NaN,NaN,False,False,1,0
3,Group stage,1.0,Tue,2021-09-14,21:00,Villarreal es,2–2,it Atalanta,12916.0,Estadio de la Cerámica,...,2021/22,2021,Villarreal,es,NaN,NaN,False,False,2,2
4,Group stage,1.0,Tue,2021-09-14,21:00,Barcelona es,0–3,de Bayern Munich,39737.0,Camp Nou,...,2021/22,2021,Barcelona,es,NaN,NaN,False,False,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1349,Play-off round,NaN,Tue,2026-08-25,21:00,Bodø/Glimt no,NaN,nl NEC Nijmegen,NaN,Aspmyra Stadion,...,2026/27,2026,Bodø/Glimt,no,NaN,NaN,False,False,<NA>,<NA>
1350,Play-off round,NaN,Wed,2026-08-26,21:00,Lyon fr,NaN,tr Fenerbahçe,NaN,Groupama Stadium,...,2026/27,2026,Lyon,fr,NaN,NaN,False,False,<NA>,<NA>
1351,Play-off round,NaN,Wed,2026-08-26,21:00,Viking no,NaN,hr Dinamo Zagreb,NaN,Lyse Arena,...,2026/27,2026,Viking,no,NaN,NaN,False,False,<NA>,<NA>
1352,Play-off round,NaN,Wed,2026-08-26,21:00,NK Celje si,NaN,sk Slovan Bratislava,NaN,Stadion Z'dežele,...,2026/27,2026,NK Celje,si,NaN,NaN,False,False,<NA>,<NA>


In [125]:
df_champions_league_test = df_champions_league[df_champions_league['Score'].str.len() >=6]

In [132]:
result_pattern = r'(?:\(\d+\)\s*)?(\d+)\s*[–-]\s*(\d+)(?:\s*\(\d+\))?'
result = df_champions_league_test['Score'].str.extract(pattern)
df_champions_league_test['Host goals'] = pd.to_numeric(result[0]).astype('Int64')
df_champions_league_test['Guest goals'] = pd.to_numeric(result[1]).astype('Int64')

In [133]:
df_champions_league_test

,Round,Wk,Day,Date,Time,Home,Score,Away,Attendance,Venue,...,season,season_start,Host,Host Country,Host penalties,Guest penalties,penalty kicks,extra time,Host goals,Guest goals
214,Third qualifying round,NaN,Tue,2021-08-10,21:00,Ludogorets bg,(4) 2–2 (1),gr Olympiacos,5118.0,Huvepharma Arena,...,2021/22,2021,Ludogorets,bg,4,1,True,True,2,2
381,First qualifying round,NaN,Tue,2022-07-12,18:30,RFS lv,(4) 2–1 (5),fi HJK Helsinki,1631.0,LNK Sporta Parks,...,2022/23,2022,RFS,lv,4,5,True,True,2,1
394,First qualifying round,NaN,Wed,2022-07-13,21:30,CFR Cluj ro,(3) 2–2 (4),am Pyunik,7017.0,Stadionul Dr. Constantin Rădulescu,...,2022/23,2022,CFR Cluj,ro,3,4,True,True,2,2
408,Second qualifying round,NaN,Tue,2022-07-26,18:30,AEK Larnaca cy,(3) 1–1 (4),dk Midtjylland,6163.0,AEK Arena - George Karapatakis,...,2022/23,2022,AEK Larnaca,cy,3,4,True,True,1,1
568,Round of 16,NaN,Tue,2024-03-12,20:00,Arsenal eng,(4) 1–0 (2),pt Porto,60257.0,Emirates Stadium,...,2023/24,2023,Arsenal,eng,4,2,True,True,1,0
571,Round of 16,NaN,Wed,2024-03-13,21:00,Atlético Madrid es,(3) 2–1 (2),it Inter,69196.0,Stadium Metropolitano,...,2023/24,2023,Atlético Madrid,es,3,2,True,True,2,1
578,Quarter-finals,NaN,Wed,2024-04-17,20:00,Manchester City eng,(3) 1–1 (4),es Real Madrid,52306.0,Etihad Stadium,...,2023/24,2023,Manchester City,eng,3,4,True,True,1,1
612,First qualifying round,NaN,Tue,2023-07-18,20:00,Zrinjski Mostar ba,(4) 2–3 (3),am Urartu,5000.0,Stadion Bijeli Brijeg,...,2023/24,2023,Zrinjski Mostar,ba,4,3,True,True,2,3
638,Second qualifying round,NaN,Wed,2023-08-02,19:00,BK Häcken se,(3) 3–3 (4),fo KÍ Klaksvík,4396.0,Bravida Arena,...,2023/24,2023,BK Häcken,se,3,4,True,True,3,3
639,Second qualifying round,NaN,Wed,2023-08-02,19:00,Genk be,(1) 2–2 (4),ch Servette FC,16280.0,Cegeka Arena,...,2023/24,2023,Genk,be,1,4,True,True,2,2


In [94]:
df_champions_league_test = df_champions_league

In [116]:
df_champions_league

,Round,Wk,Day,Date,Time,Home,Score,Away,Attendance,Venue,...,Match Report,Notes,competition_phase,season,season_start,Host,Host Country,Host penalties,Guest penalties,penalty kicks
0,Group stage,1.0,Tue,2021-09-14,18:45,Sevilla es,1–1,at RB Salzburg,18373.0,Estadio Ramón Sánchez Pizjuán,...,Match Report,NaN,main stage,2021/22,2021,Sevilla,es,NaN,NaN,False
1,Group stage,1.0,Tue,2021-09-14,18:45,Young Boys ch,2–1,eng Manchester Utd,31120.0,Wankdorfstadion,...,Match Report,NaN,main stage,2021/22,2021,Young Boys,ch,NaN,NaN,False
2,Group stage,1.0,Tue,2021-09-14,20:00,Chelsea eng,1–0,ru Zenit,39252.0,Stamford Bridge,...,Match Report,NaN,main stage,2021/22,2021,Chelsea,eng,NaN,NaN,False
3,Group stage,1.0,Tue,2021-09-14,21:00,Villarreal es,2–2,it Atalanta,12916.0,Estadio de la Cerámica,...,Match Report,NaN,main stage,2021/22,2021,Villarreal,es,NaN,NaN,False
4,Group stage,1.0,Tue,2021-09-14,21:00,Barcelona es,0–3,de Bayern Munich,39737.0,Camp Nou,...,Match Report,NaN,main stage,2021/22,2021,Barcelona,es,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1349,Play-off round,NaN,Tue,2026-08-25,21:00,Bodø/Glimt no,NaN,nl NEC Nijmegen,NaN,Aspmyra Stadion,...,Head-to-Head,Leg 2 of 2,qualification,2026/27,2026,Bodø/Glimt,no,NaN,NaN,False
1350,Play-off round,NaN,Wed,2026-08-26,21:00,Lyon fr,NaN,tr Fenerbahçe,NaN,Groupama Stadium,...,Head-to-Head,Leg 2 of 2,qualification,2026/27,2026,Lyon,fr,NaN,NaN,False
1351,Play-off round,NaN,Wed,2026-08-26,21:00,Viking no,NaN,hr Dinamo Zagreb,NaN,Lyse Arena,...,Head-to-Head,Leg 2 of 2,qualification,2026/27,2026,Viking,no,NaN,NaN,False
1352,Play-off round,NaN,Wed,2026-08-26,21:00,NK Celje si,NaN,sk Slovan Bratislava,NaN,Stadion Z'dežele,...,Head-to-Head,Leg 2 of 2,qualification,2026/27,2026,NK Celje,si,NaN,NaN,False


In [99]:
penalties = df_champions_league_test['Score'].str.findall(r'\((\d+)\)')

In [100]:
print(penalties)

0        []
1        []
2        []
3        []
4        []
       ... 
1349    NaN
1350    NaN
1351    NaN
1352    NaN
1353    NaN
Name: Score, Length: 1354, dtype: object


In [105]:
df_champions_league_test['Host penalties'] = penalties.apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else None)
df_champions_league_test['Guest penalties'] = penalties.apply(lambda x: x[-1] if isinstance(x, list) and len(x) >= 2 else None)

In [106]:
df_champions_league_test

,Round,Wk,Day,Date,Time,Home,Score,Away,Attendance,Venue,Referee,Match Report,Notes,competition_phase,season,season_start,Host,Host Country,Host penalties,Guest penalties
214,Third qualifying round,NaN,Tue,2021-08-10,21:00,Ludogorets bg,(4) 2–2 (1),gr Olympiacos,5118.0,Huvepharma Arena,Orel Grinfeeld,Match Report,Leg 2 of 2; Ludogorets won; Ludogorets won a p...,qualification,2021/22,2021,Ludogorets,bg,4,1
381,First qualifying round,NaN,Tue,2022-07-12,18:30,RFS lv,(4) 2–1 (5),fi HJK Helsinki,1631.0,LNK Sporta Parks,David Fuxman,Match Report,Leg 2 of 2; HJK Helsinki won; HJK Helsinki won...,qualification,2022/23,2022,RFS,lv,4,5
394,First qualifying round,NaN,Wed,2022-07-13,21:30,CFR Cluj ro,(3) 2–2 (4),am Pyunik,7017.0,Stadionul Dr. Constantin Rădulescu,Michael Fabbri,Match Report,Leg 2 of 2; Pyunik won; Pyunik won a penalty s...,qualification,2022/23,2022,CFR Cluj,ro,3,4
408,Second qualifying round,NaN,Tue,2022-07-26,18:30,AEK Larnaca cy,(3) 1–1 (4),dk Midtjylland,6163.0,AEK Arena - George Karapatakis,István Vad,Match Report,Leg 2 of 2; Midtjylland won; Midtjylland won a...,qualification,2022/23,2022,AEK Larnaca,cy,3,4
568,Round of 16,NaN,Tue,2024-03-12,20:00,Arsenal eng,(4) 1–0 (2),pt Porto,60257.0,Emirates Stadium,Clément Turpin,Match Report,Leg 2 of 2; Arsenal won; Arsenal won a penalty...,main stage,2023/24,2023,Arsenal,eng,4,2
571,Round of 16,NaN,Wed,2024-03-13,21:00,Atlético Madrid es,(3) 2–1 (2),it Inter,69196.0,Stadium Metropolitano,Szymon Marciniak,Match Report,Leg 2 of 2; Atlético Madrid won; Atlético Madr...,main stage,2023/24,2023,Atlético Madrid,es,3,2
578,Quarter-finals,NaN,Wed,2024-04-17,20:00,Manchester City eng,(3) 1–1 (4),es Real Madrid,52306.0,Etihad Stadium,Daniele Orsato,Match Report,Leg 2 of 2; Real Madrid won; Real Madrid won a...,main stage,2023/24,2023,Manchester City,eng,3,4
612,First qualifying round,NaN,Tue,2023-07-18,20:00,Zrinjski Mostar ba,(4) 2–3 (3),am Urartu,5000.0,Stadion Bijeli Brijeg,Peter Kjaesgaard,Match Report,Leg 2 of 2; Zrinjski Mostar won; Zrinjski Most...,qualification,2023/24,2023,Zrinjski Mostar,ba,4,3
638,Second qualifying round,NaN,Wed,2023-08-02,19:00,BK Häcken se,(3) 3–3 (4),fo KÍ Klaksvík,4396.0,Bravida Arena,Andy Madley,Match Report,Leg 2 of 2; KÍ Klaksvík won; KÍ Klaksvík won a...,qualification,2023/24,2023,BK Häcken,se,3,4
639,Second qualifying round,NaN,Wed,2023-08-02,19:00,Genk be,(1) 2–2 (4),ch Servette FC,16280.0,Cegeka Arena,Novak Simović,Match Report,Leg 2 of 2; Servette FC won; Servette FC won a...,qualification,2023/24,2023,Genk,be,1,4


In [92]:
print(df_champions_league_test['Score'].str.index("("))

214     0
381     0
394     0
408     0
568     0
571     0
578     0
612     0
638     0
639     0
656     0
664     0
856     0
862     0
890     0
891     0
902     0
1165    0
1183    0
1243    0
1255    0
1309    0
1310    0
1312    0
Name: Score, dtype: int64
